# Task 2 & 3 – Recommendation Models and Evaluation
**INFO345 – RateBeer dataset**

We build and evaluate the collaborative filtering recommenders from the course (Lecture 3–5, Lab 2 and Lab 3) on our RateBeer data, using **Scikit Surprise** and **5-fold cross validation**. Models are compared on rating prediction accuracy with **RMSE** and **MAE**.

| Model in the table | Surprise class | Type |
|---|---|---|
| SVD | `SVD` | Matrix factorization |
| SVD++ | `SVDpp` | Matrix factorization (+ implicit feedback) |
| KNN-Basic | `KNNBasic` | Neighbourhood-based CF |
| KNN-Baseline | `KNNBaseline` | Neighbourhood-based CF with baseline ratings |
| Baseline | `BaselineOnly` | Global mean + user bias + item bias |
| Random | `NormalPredictor` | Random rating from the rating distribution |

Requires `01_data_analysis.ipynb` to have been run first (it writes `filtered.parquet`).

## Loading the data

In [1]:
import pandas as pd
import numpy as np

DATA = ".."   # the 'RateBeer Dataset' folder

# Ratings after the filtering in Task 1 (2012–2017, users >= 10 ratings, beers >= 5 ratings)
ratings = pd.read_parquet(f"{DATA}/filtered.parquet")
ratings = ratings[['user_id', 'beer_id', 'rating']]
ratings.head()

,user_id,beer_id,rating
0,442761,105273,1.9
1,288889,105273,1.6
2,250510,105273,1.5
3,122778,105273,1.9
4,227834,105273,1.9


<details>
<summary><b>💡 What's going on here?</b></summary>

We load the ratings that were filtered in the first notebook and keep only the three columns Surprise needs: **user**, **item** (beer) and **rating**.

</details>

SVD++ is very slow for users with thousands of ratings (our most active user has over 20,000). To make all models trainable on a laptop, we keep **at most 100 randomly chosen ratings per user**. All users are kept, and very active users no longer dominate the data.

In [2]:
print("Ratings per user before sampling:")
print(ratings.groupby('user_id').size().describe().round(1))

# keep at most 100 random ratings per user (fixed random_state for reproducibility)
ratings = ratings.sample(frac=1, random_state=42).groupby('user_id').head(100)

num_users = ratings['user_id'].nunique()
num_beers = ratings['beer_id'].nunique()
num_ratings = len(ratings)
density = num_ratings / (num_users * num_beers)

print("\nNumber of users:", num_users)
print("Number of beers rated:", num_beers)
print("Number of ratings:", num_ratings)
print("Data density: {:.4%}".format(density))
print("Data sparsity: {:.4%}".format(1 - density))

Ratings per user before sampling:
count    10368.0
mean       363.1
std       1022.8
min         10.0
25%         18.0
50%         49.0
75%        243.0
max      20670.0
dtype: float64



Number of users: 10368
Number of beers rated: 88828
Number of ratings: 592205
Data density: 0.0643%
Data sparsity: 99.9357%


<details>
<summary><b>💡 What's going on here?</b></summary>

**Why we take a sample:** one of the models, SVD++, gets extremely slow for users with many ratings (our most active user has over 20,000). On the full 3.7 million ratings it would not finish on a laptop.

So for every user we keep at most **100 randomly chosen ratings**. `sample(frac=1, random_state=42)` shuffles the rows (with a fixed seed, so we get the same sample every time), and `groupby('user_id').head(100)` keeps the first 100 per user. All 10,368 users are kept; only very active users lose some ratings, which also stops them from dominating the results.

We then print the size of the data and its density/sparsity, as in Lab 3.

</details>

## Model based approaches with Surprise

In [3]:
#Import the required classes and methods from the surprise library
from surprise import SVD, SVDpp, KNNBasic, KNNBaseline, BaselineOnly, NormalPredictor
from surprise import Reader, Dataset
from surprise.model_selection import cross_validate, KFold
from tabulate import tabulate
import time
import datetime

#Define a Reader object
#The Reader object helps in parsing the dataframe containing ratings
#RateBeer ratings go from 0.5 to 5
reader = Reader(rating_scale=(0.5, 5))

#Create the dataset to be used for building the filter
data = Dataset.load_from_df(ratings[['user_id', 'beer_id', 'rating']], reader)

<details>
<summary><b>💡 What's going on here?</b></summary>

We import the recommender algorithms from **Scikit Surprise**, the library used in Lab 2 and Lab 3.

- **`Reader`** tells Surprise the rating scale. RateBeer ratings go from **0.5 to 5** (MovieLens in the labs used 1–5).
- **`Dataset.load_from_df`** turns our pandas table into a Surprise dataset with the columns user, item, rating.

</details>

### K-fold cross validation
We use 5-fold cross validation, which randomly splits the data into 5 equal parts. Each part is used once as test set while the other 4 parts are used for training, and the evaluation scores are averaged over the 5 folds.

In [4]:
#let's make the random a bit less random
np.random.seed(0)
# split data into 5 folds
kf = KFold(n_splits=5, random_state=0)

<details>
<summary><b>💡 What's going on here?</b></summary>

**5-fold cross validation** (Lecture 5): the data is split randomly into 5 equal parts. Each model is trained 5 times – each time on 4 parts (80%) – and tested on the remaining part (20%). Every rating is used for testing exactly once, and the final score is the average over the 5 rounds. This gives a more reliable result than a single train/test split.

`random_state=0` fixes how the data is split, so every model is tested on exactly the same folds and the results can be reproduced.

</details>

In [5]:
#The recommenders and their titles
recommenders = [SVD(), SVDpp(), KNNBasic(), KNNBaseline(), BaselineOnly(), NormalPredictor()]
titles = ['SVD', 'SVD++', 'KNN-Basic', 'KNN-Baseline', 'Baseline', 'Random']

<details>
<summary><b>💡 What's going on here?</b></summary>

The six models we compare, with the names used in the results table:

| Name | What it does |
|---|---|
| **SVD** | Matrix factorization: learns a short list of hidden "taste numbers" for every user and every beer, and predicts a rating from how well they match. Also learns how generous each user is and how good each beer is in general. |
| **SVD++** | SVD, but it also uses *which* beers a user has rated (not only how they rated them). Usually a bit more accurate, but much slower. |
| **KNN-Basic** | Finds the users most similar to you (k nearest neighbours) and predicts your rating as their average rating for the beer. |
| **KNN-Baseline** | Like KNN-Basic, but first adjusts for how generous each user is and how good each beer is in general, and uses the neighbours only for the rest. |
| **Baseline** | Only the general part: overall average + how generous the user is + how good the beer is. No personal taste. |
| **Random** | Picks a random rating from the overall rating distribution. The lower limit every real model should beat. |

</details>

In [6]:
table = []
for algo, title in zip(recommenders, titles):
    start = time.time()

    # cross validate on 5 folds on two evaluation metrics
    result = cross_validate(algo, data, ['rmse', 'mae'], kf)

    cv_time = str(datetime.timedelta(seconds=int(time.time() - start)))
    mean_rmse = '{:.3f}'.format(np.mean(result['test_rmse']))
    mean_mae = '{:.3f}'.format(np.mean(result['test_mae']))

    table.append([title, mean_rmse, mean_mae, cv_time])
    print(title, "done in", cv_time)

SVD done in 0:01:44


SVD++ done in 0:07:04


Computing the msd similarity matrix...
Done computing similarity matrix.


Computing the msd similarity matrix...
Done computing similarity matrix.


Computing the msd similarity matrix...
Done computing similarity matrix.


...

Estimating biases using als...


Estimating biases using als...


Estimating biases using als...


Baseline done in 0:00:51


Random done in 0:00:28


<details>
<summary><b>💡 What's going on here?</b></summary>

For every model, `cross_validate` runs the 5-fold cross validation and returns the test scores of each fold. We store the **average RMSE**, **average MAE** and the time it took.

- **MAE** (Mean Absolute Error): on average, how many rating points the prediction is off.
- **RMSE** (Root Mean Squared Error): similar, but big mistakes count more.

For both, **lower is better** and **0 would be perfect**. SVD++ takes by far the longest (about 7 minutes); Random and Baseline take under a minute.

</details>

### Results

In [7]:
header = ['Recommenders', 'Pred Accuracy [RMSE]', 'Pred Accuracy [MAE]']
print(tabulate([row[:3] for row in table], header, tablefmt="pipe"))

| Recommenders   |   Pred Accuracy [RMSE] |   Pred Accuracy [MAE] |
|:---------------|-----------------------:|----------------------:|
| SVD            |                  0.533 |                 0.392 |
| SVD++          |                  0.52  |                 0.381 |
| KNN-Basic      |                  0.574 |                 0.419 |
| KNN-Baseline   |                  0.533 |                 0.39  |
| Baseline       |                  0.532 |                 0.39  |
| Random         |                  1.001 |                 0.79  |


<details>
<summary><b>💡 What's going on here?</b></summary>

The results table, in the same format as in the lecture.

**How to read it:** the numbers are **errors in rating points** on the 0.5–5 scale, not percentages. **Lower is better.** For example, SVD++ has MAE 0.381: its predictions are on average about 0.4 points away from the real rating.

**Is that good?** Always guessing the average rating would give RMSE 0.712. SVD++ (0.520) has about **27% lower error** than that, while Random (1.001) is worse than just guessing the average.

**Why are the numbers lower than in the lecture (≈0.93 for SVD)?** MovieLens uses whole stars from 1 to 5, which are much more spread out. Our ratings are in steps of 0.1 and mostly lie between 3 and 4, so errors are naturally smaller. Results can therefore only be compared *between models on the same data*, not with the MovieLens numbers.

</details>

In [8]:
# The same table including the time used for the 5-fold cross validation
print(tabulate(table, header + ['Time'], tablefmt="pipe"))

| Recommenders   |   Pred Accuracy [RMSE] |   Pred Accuracy [MAE] | Time    |
|:---------------|-----------------------:|----------------------:|:--------|
| SVD            |                  0.533 |                 0.392 | 0:01:44 |
| SVD++          |                  0.52  |                 0.381 | 0:07:04 |
| KNN-Basic      |                  0.574 |                 0.419 | 0:02:19 |
| KNN-Baseline   |                  0.533 |                 0.39  | 0:02:33 |
| Baseline       |                  0.532 |                 0.39  | 0:00:51 |
| Random         |                  1.001 |                 0.79  | 0:00:28 |


<details>
<summary><b>💡 What's going on here?</b></summary>

The same table with the time each model needed for the full 5-fold cross validation. This is a trade-off worth mentioning in the report: SVD++ is the most accurate, but it is about **4× slower than SVD** for a small improvement (RMSE 0.520 vs 0.533).

</details>

#### Which recommendation method performs best according to each evaluation metric?
*(Discuss the results here.)*

<details>
<summary><b>💡 Suggested interpretation</b></summary>

- **Best on both metrics: SVD++** (RMSE 0.520, MAE 0.381), because it also uses *which* beers each user has rated.
- **SVD, KNN-Baseline and Baseline are almost tied** (RMSE ≈ 0.53). Since the simple Baseline – which only knows how generous each user is and how good each beer is in general – is as good as SVD, most of what can be predicted comes from these general effects. Personal taste adds only a little on top.
- **KNN-Basic is clearly weaker** (RMSE 0.574): it uses raw neighbour ratings without adjusting for generous/strict users, which adds noise. KNN-Baseline fixes exactly that.
- **Random is far behind** (RMSE 1.001), so all real models have learned something useful.
- **Speed vs. accuracy:** SVD++ needs about 4× the time of SVD for a 2.5% lower RMSE.

</details>

<details>
<summary><b>💡 Suggested interpretation</b></summary>

- **Best on both metrics: SVD++** (RMSE 0.520, MAE 0.381), because it also uses *which* beers each user has rated.
- **SVD, KNN-Baseline and Baseline are almost tied** (RMSE ≈ 0.53). Since the simple Baseline – which only knows how generous each user is and how good each beer is in general – is as good as SVD, most of what can be predicted comes from these general effects. Personal taste adds only a little on top.
- **KNN-Basic is clearly weaker** (RMSE 0.574): it uses raw neighbour ratings without adjusting for generous/strict users, which adds noise. KNN-Baseline fixes exactly that.
- **Random is far behind** (RMSE 1.001), so all real models have learned something useful.
- **Speed vs. accuracy:** SVD++ needs about 4× the time of SVD for a 2.5% lower RMSE.

</details>